# Vietnamese Receipt EDA

## §0 Setup & Data Acquisition

In [ ]:
import os
import random
import hashlib
import json
import yaml
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

# --- Project-local cache dirs (gitignored) ---
PROJECT_ROOT = Path(".").resolve()
DATASETS_DIR = PROJECT_ROOT / "datasets"
os.environ["KAGGLEHUB_CACHE"] = str(DATASETS_DIR / "kagglehub")
os.environ["HF_HOME"] = str(DATASETS_DIR / "huggingface")
(DATASETS_DIR / "kagglehub").mkdir(parents=True, exist_ok=True)
(DATASETS_DIR / "huggingface").mkdir(parents=True, exist_ok=True)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Output dir (recreate from scratch each run) ---
OUT = Path("eda_outputs")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

# --- Download MC-OCR from Kaggle ---
# NOTE: VQA (`5CD-AI/Viet-OCR-VQA-flash2`) is deferred — gated access pending manual review.
import kagglehub
print("Downloading MC-OCR from Kaggle (cached under ./datasets/kagglehub/)...")
mcocr_path = Path(kagglehub.dataset_download("domixi1989/vietnamese-receipts-mc-ocr-2021"))
print(f"MC-OCR unpacked to: {mcocr_path}")

# Reproducibility hash for MC-OCR (Kaggle has no clean revision)
def tree_hash(path):
    h = hashlib.sha256()
    for p in sorted(path.rglob("*")):
        if p.is_file():
            h.update(str(p.relative_to(path)).encode())
            h.update(str(p.stat().st_size).encode())
    return h.hexdigest()
MCOCR_TREE_HASH = tree_hash(mcocr_path)
print(f"MC-OCR tree hash (size-based): {MCOCR_TREE_HASH[:16]}...")

# --- MC-OCR layout ---
# Confirmed via prior inspection of versions/17/ archive:
#   CSV:        mcocr_train_df.csv at the archive root
#   Image dir:  data0.7/data0.7/
#   Filename:   img_id column
mcocr_csv = mcocr_path / "mcocr_train_df.csv"
mcocr_img_dir = mcocr_path / "data0.7" / "data0.7"
print(f"MC-OCR CSV: {mcocr_csv}")
print(f"MC-OCR image dir: {mcocr_img_dir}")
mcocr_df = pd.read_csv(mcocr_csv)
print(f"MC-OCR CSV columns: {list(mcocr_df.columns)}")
print(f"MC-OCR CSV head:\n{mcocr_df.head(2).to_string()}")

def iter_mcocr():
    """Yield (example_id, image_path_str, annotation_dict)."""
    for _, row in mcocr_df.iterrows():
        name = str(row["img_id"])
        img_path = mcocr_img_dir / name
        if not img_path.exists():
            continue
        yield (name, str(img_path), row.to_dict())

# --- Materialize counts (cheap) ---
MCOCR_COUNT = sum(1 for _ in iter_mcocr())
print(f"\nMC-OCR examples reachable: {MCOCR_COUNT}")

## §1 Schema & Field Analysis → schema.json

In [ ]:
def annotation_pairs():
    """Yield (dataset_name, ann_dict) for MC-OCR. VQA half deferred (see scope amendment)."""
    for _id, _img, ann in iter_mcocr():
        yield ("mc_ocr", ann if isinstance(ann, dict) else dict(ann))

field_obs = {}
totals = Counter()

for ds, ann in annotation_pairs():
    totals[ds] += 1
    for k, v in ann.items():
        rec = field_obs.setdefault(k, {"datasets": set(), "present_per_ds": Counter(), "samples": []})
        rec["datasets"].add(ds)
        if v is not None and v != "" and v != [] and not (isinstance(v, float) and pd.isna(v)):
            rec["present_per_ds"][ds] += 1
            if len(rec["samples"]) < 5:
                rec["samples"].append(str(v)[:120])

def infer_type(samples):
    if not samples:
        return "string"
    if all(s.replace(",", "").replace(".", "").replace("-", "").replace(" ", "").isdigit()
           for s in samples if s):
        return "number"
    if any(("/" in s and len(s) <= 12) or ("-" in s and len(s) == 10) for s in samples):
        return "date"
    if any(s.lstrip().startswith("[") for s in samples):
        return "array"
    return "string"

fields = []
for name, rec in sorted(field_obs.items()):
    denom = sum(totals[d] for d in rec["datasets"])
    coverage = sum(rec["present_per_ds"].values()) / denom if denom else 0.0
    fields.append({
        "name": name,
        "type": infer_type(rec["samples"]),
        "source_datasets": sorted(rec["datasets"]),
        "coverage_rate": round(coverage, 3),
        "format_observations": rec["samples"][:5],
    })

schema = {"fields": fields, "totals": dict(totals)}
(OUT / "schema.json").write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"§1 wrote schema.json with {len(fields)} fields")
for f in fields[:10]:
    print(f"  {f['name']:30s} type={f['type']:7s} cov={f['coverage_rate']:.2f} ds={f['source_datasets']}")

## §2 Image Characteristics → training_caps.yaml (resolution)

## §3 Text Characteristics → normalization_rules.yaml + training_caps.yaml (seq_length)

## §4 Visual Quality (Automated) → augmentation.yaml

## §5 VQA Structure → prompt_templates.yaml

## §6 Cross-Dataset Comparison → training_strategy.yaml

## §7 Summary & Sanity Checks

In [ ]:
import os, json, yaml
from pathlib import Path

OUT = Path("eda_outputs")

def validate_section_0():
    """§0 produced MC-OCR iterator with non-zero count."""
    assert "MCOCR_COUNT" in globals(), "§0 did not run; MCOCR_COUNT undefined"
    assert MCOCR_COUNT > 0, "MC-OCR yielded zero examples"
    print(f"  §0 OK — MC-OCR: {MCOCR_COUNT}")

def validate_section_1():
    """§1 emits schema.json with non-empty fields list and valid coverage rates."""
    p = OUT / "schema.json"
    assert p.exists(), f"{p} missing"
    schema = json.loads(p.read_text(encoding="utf-8"))
    assert "fields" in schema, "schema.json has no 'fields' key"
    assert len(schema["fields"]) > 0, "schema.json fields list is empty"
    for f in schema["fields"]:
        for key in ("name", "type", "source_datasets", "coverage_rate", "format_observations"):
            assert key in f, f"field missing key '{key}': {f}"
        assert 0.0 <= f["coverage_rate"] <= 1.0, f"coverage_rate out of range: {f}"
        assert f["type"] in ("string", "number", "date", "array"), f"unknown type: {f}"
    print(f"  §1 OK — {len(schema['fields'])} fields cataloged")


def run_all_validations():
    print("Running sanity checks...")
    validate_section_0()
    validate_section_1()
    print("All sanity checks passed.")

run_all_validations()